# Part 5: The Analyst Report

After you have successfully deployed your pipeline and run the **Burst** profile (500 messages) in the test apparatus, you need to extract the results and answer a few questions.

We use `boto3` to scan the DynamoDB table, handling pagination automatically, and convert the results into standard Python dictionaries and floats.

## Setup: Configure Your Student ID
Replace `YOURID` below with the exact student ID you used for deployment.

In [3]:
%pip install boto3
STUDENT_ID = "mlara"
TABLE_NAME = f"adflow-mlara-results"
REGION = "us-east-1"
print(f"Target Table: {TABLE_NAME}")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Target Table: adflow-mlara-results


## Step 1: Export Data from DynamoDB
This cell connects to your DynamoDB table, downloads all records, and converts the Decimal values back to standard floats.

In [5]:
import boto3
from decimal import Decimal
from collections import Counter

# Note: This uses your active AWS credentials (from `aws configure` or exported environment variables)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

results = []
response = table.scan()
results.extend(response.get("Items", []))

# Handle pagination if the table has more than 1 MB of data
while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    results.extend(response.get("Items", []))

print(f"\nLoaded {len(results)} records from DynamoDB.")

# Convert Decimal types to Python floats for easier math/plotting
for item in results:
    for key in ["winning_bid_amount", "winning_score", "score_margin"]:
        if key in item and isinstance(item[key], Decimal):
            item[key] = float(item[key])

if results:
    print("\nSample record:")
    print(results[0])


Loaded 500 records from DynamoDB.

Sample record:
{'processed_at': '2026-03-24T14:45:41.112754Z', 'opportunity_id': 'bff00546-23d3-44ef-9616-207f4c52a9a6', 'winning_score': 8.69375, 'score_margin': 1.7437500000000004, 'winning_advertiser_id': 'adv_energy_01', 'winning_bid_amount': 5.35, 'content_category': 'sports'}


## Section 1: Pipeline Evidence
Print the total records and a quick count of auction wins per advertiser across the entire dataset to prove your pipeline successfully routed messages.

In [6]:
# TODO: Print the total number of records
print(f"Total pipeline records: {len(results)}")

# TODO: Compute and print the auction wins per advertiser (overall)
# Hint: Use collections.Counter on the 'winning_advertiser_id' field

from collections import Counter

# Auction wins per advertiser (overall)
overall_counts = Counter([r["winning_advertiser_id"] for r in results])

print("\nAuction wins per advertiser (overall):")
for advertiser, count in overall_counts.most_common():
    print(f"{advertiser}: {count}")

Total pipeline records: 500

Auction wins per advertiser (overall):
adv_auto_01: 67
adv_fintech_01: 59
adv_insurance_01: 56
adv_travel_01: 45
adv_streaming_01: 44
adv_fastfood_01: 34
adv_energy_01: 29
adv_auto_02: 26
adv_sportswear_01: 25
adv_fastfood_02: 21
adv_insurance_02: 18
adv_beauty_01: 14
adv_gaming_01: 11
adv_fintech_02: 11
adv_telecom_01: 10
adv_travel_02: 10
adv_energy_02: 6
adv_streaming_02: 6
adv_sportswear_02: 3
adv_retail_01: 3
adv_gaming_02: 1
adv_beauty_02: 1


**Evidence Requirement:** Don't forget to push a screenshot of the **Test Apparatus** (showing a completed Burst run) to a `screenshots/` directory in this repo when submitting.

---
## Q1: Results Analysis

**Question:** Which advertiser won the most auctions overall? Which advertiser won the most in the `sports` content category specifically? Why do the overall and sports-specific rankings differ? Explain in 2–3 sentences, referencing the relevance multiplier table.

In [8]:
# TODO: Find the top winner in the 'sports' category

from collections import Counter
# Filter only sports records
sports_results = [r for r in results if r["content_category"] == "sports"]

# Count wins
sports_counts = Counter([r["winning_advertiser_id"] for r in sports_results])

print("Sports winners:")
print(sports_counts.most_common(5))


Sports winners:
[('adv_auto_01', 24), ('adv_energy_01', 18), ('adv_sportswear_01', 17), ('adv_fintech_01', 17), ('adv_travel_01', 11)]


**Your Answer (Q1):**

### Q1: Results Analysis

The advertiser that won the most auctions overall is **adv_auto_01**, with **67** wins.
In the `sports` content category, the advertiser that won the most auctions is **adv_auto_01**, with **24** wins.

In this dataset, adv_auto_01 remains dominant even in sports content, suggesting that its bid amounts are consistently high enough to offset the relevance advantage of sports-related advertisers.

The overall and sports-specific rankings differ due to the relevance multiplier used in the scoring function. In sports content, advertisers whose categories match the content (such as sportswear or energy drink) receive higher multipliers, which increases their effective score even if their raw bid is lower. As a result, advertisers that are not dominant overall can perform better within specific content categories where they have a relevance advantage.


---
## Q2: Code Reflection

Answer **one** of the following (your choice):
 
* **Option A (Scale & Limits):** The test apparatus sent messages in small batches. If traffic suddenly spiked from 10 opportunities a second to 10,000 a second, what specific components of our current pipeline (SQS limits, Lambda concurrency, DynamoDB throughput) would become bottlenecks first, and what AWS settings would you adjust to handle the load?
* **Option B (The Distributed Process):** Writing code for an event-driven, queue-based pipeline is very different from writing a single local script. What was the most challenging part of getting SQS, Lambda, and DynamoDB to communicate correctly, or the most confusing bug you encountered, and what did it teach you about distributed architecture?

A well-argued two-paragraph response is sufficient for either option.

**Your Answer (Q2):**

### Q2 — Code Reflection 

The most challenging part of this project was understanding how the different AWS components (SQS, Lambda, and DynamoDB) interact within an event-driven architecture. Unlike a local script where execution is linear and immediate, this system is asynchronous, meaning that messages are processed independently and timing is not deterministic. This made it more difficult to trace the flow of data and identify where issues were occurring.

During the development process, I performed several rounds of testing because I initially believed that my Lambda handler implementation might be inefficient, especially when I observed that latency was not consistently within the green range in the test apparatus. After reviewing CloudWatch logs and adding debug statements and going back sveral iterations, I realized that the main cause of the higher latency was not the code itself, but external factors such as SQS polling delays and Lambda cold starts. This highlighted an important lesson about distributed systems: performance bottlenecks are often not in the code logic, but in the communication and coordination between components. Understanding this distinction was key to correctly interpreting system performance.